## Simulated survey:
1. pool of R respondents each with preference vector beta_r
2. run batches of T tasks with R respondents and m expected contrasts/task
3. before first batch: compute + normalize sampling weights to assign baseline p_flip
4. afer each batch: pool comparisons + apply move-on rule for significant features to get updated p_flip

In [27]:
import pandas as pd
import numpy as np
from numpy.typing import NDArray

In [28]:
import sys

In [29]:
import statsmodels.api as sm

In [30]:
from scipy.stats import norm

In [31]:
rand = np.random.default_rng(seed=42)

In [32]:
# global vars for
# - list of features/attributes
# - number of features/attributes
# - number of tasks/batch
# - number of respondents/batch
# - (sigma)^2: heterogeneity in respondent preferences
# - power-law concentration
# - expected number of flipped features/task

In [146]:
FEATURES = ["setting:rural", "setting:urban", "setting:government", 
            "expression_angry", "expression_determined", "expression_sad", "expression_neutral", 
            "race_white", "race_black", "race_hispanic", "race_asian", 
            "age_old", "age_young", 
            "sign_amazon", "sign_google", "sign_microsoft", 
            "focal_single"]
P = len(FEATURES)
T = 10
R = 150
RES_VAR = 0.10
gamma = 1.2
m = 2

In [135]:
# each mutually exclusive category mapped to the positions of its options, i.e. {'setting': [0, 1, 2]}
    # expects format [category]:[option] in provided list of features
CATS = {}
for i, f in enumerate(FEATURES):
    if ":" in f:
        CATS.setdefault(f.split(":", 1)[0].strip(), []).append(i)
        
# positions of binary features
BIN = [i for i, f in enumerate(FEATURES) if ":" not in f]

# positions of units that can be contrasted, 
    # i.e. [[0, 1, 2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16]]
UNITS = list(CATS.values()) + [[i] for i in BIN]   
N_CAT = len(CATS)
U = len(UNITS)

# reference position (first appearance of category)
REF = [cols[0] for cols in CATS.values()]      

# which columns to fit on, for a category with k options, keep k-1 columns (throw out first option)
EST = [i for i in range(P) if i not in REF]

In [121]:
EST

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

### Pool of simulated respondents

In [35]:
# get B_true baseline with index-based power-law decay: 

In [128]:
k = np.arange(1, P + 1)

In [129]:
C = 1 / np.sum(k ** (-gamma))

In [130]:
B_true = C * k ** (-gamma)

In [131]:
B_true

array([0.36088852, 0.15708585, 0.09656674, 0.06837559, 0.05231293,
       0.04203311, 0.03493457, 0.0297622 , 0.02583938, 0.02277053,
       0.02030962, 0.01829598, 0.01662038, 0.01520616, 0.01399792,
       0.01295475, 0.01204576])

In [132]:
var_matrix = RES_VAR * np.eye(P)

In [133]:
# add heterogeneity and generate full pool of respondents 

In [134]:
B = rand.multivariate_normal(mean=B_true, cov=var_matrix, size=R)
B.shape

(150, 17)

### "Images" (1 category with mutually exclusive features)

In [84]:
features = pd.read_csv("images/data-center-protests/feature_combinations.csv")

In [85]:
feature_cols = features.columns[1:]
features["id"] = features[feature_cols].astype(str).agg("".join, axis=1)

In [49]:
features

,id,setting_rural,setting_urban,setting_government,expression_angry,expression_determined,expression_sad,expression_neutral,race_white,race_black,race_hispanic,race_asian,age_old,age_young,sign_amazon,sign_google,sign_microsoft,focal_single
0,10000000000000000,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,10000000000000001,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,10000000000000010,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,10000000000000011,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
4,10000000000000100,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49147,00111111111111011,0,0,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1
49148,00111111111111100,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0
49149,00111111111111101,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1
49150,00111111111111110,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0


### Helpers

In [98]:
def flip_probs(preference):
    if (np.sum(np.abs(preference)) > 0):
        weights = np.abs(preference) / np.sum(np.abs(preference))
    else:
        weights = np.full(U, 1/U)
    p_flip = m * weights

    # cap at 1, redistribute over remaining features if exceed 
    while (p_flip > 1).any():
        pinned = p_flip >= 1
        p_flip[pinned] = 1.0
        free = ~pinned & (p_flip > 0)
        if not free.any():
            break
        p_flip[free] *= (m - pinned.sum()) / p_flip[free].sum()
    return p_flip

In [99]:
def move_on(S_b, preference):
    raw_weights = np.abs(preference).copy()
    raw_weights[list(S_b)] = 0
    return flip_probs(raw_weights)

In [150]:
def make_img2(img1, flips):
    img2 = img1.copy()
    for g in np.flatnonzero(flips):
        cols = UNITS[g]
        if g < N_CAT:                              
            current = cols[np.argmax(img1[cols])]
            img2[cols] = 0
            img2[rand.choice([c for c in cols if c != current])] = 1
        else:                                      
            img2[cols[0]] ^= 1
    return img2

In [ ]:
# two options for turning 17-feature Beta into 15-unit weights (because there are 17 columns but 15 big categories):

In [144]:
# use difference between highest and lowest Beta within each category
def unit_weights(preference):
    return np.array([preference[cols].max() - preference[cols].min() if g < N_CAT
                     else abs(preference[cols[0]]) for g, cols in enumerate(UNITS)])

In [142]:
# use 
def unit_weights(preference):
    out = []
    for g, cols in enumerate(UNITS):
        b = preference[cols]
        if g < N_CAT:
            out.append(np.abs(b[:, None] - b[None, :])[np.triu_indices(len(b), 1)].mean())
        else:
            out.append(abs(b[0]))
    return np.array(out)

### Running batches (batch-adaptive, significance-gated move-on strategy)

In [101]:
# baseline example:
p_flip = flip_probs(B_true)
rand.binomial(1, p_flip, size=U)

array([1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0])

In [148]:
def run_batches(init_preferences, B_population, data, num_B):
    comparisons = []
    choices = []

    # initial contrast probablities
    p_flip = flip_probs(unit_weights(init_preferences))
    
    for b in range(num_B):
        # generate T tasks for each respondent with current contrast prob
        for r in B_population:
            for t in range(T):
                # for current csv setup with id column describing feature vector (i.e. '10001000101000101')
                # note no mutually exclusive features in test set (with full 2^12 combination) for simplicity 
                # realistically would need to handle when users upload data without all combinations
                img1 = np.fromiter(rand.choice(data["id"].values), dtype=int)
                flips = rand.binomial(1, p_flip)
                #img2 = img1 ^ flips
                img2 = make_img2(img1, flips)
    
                # r "chooses" img1 or img2
                d = img1 - img2
                utility_diff = np.dot(d, r)
                p_img1 = 1 / (1 + np.exp(-utility_diff))
                choice = rand.binomial(1, p_img1)
    
                # keep track of comparisons
                comparisons.append(d[EST])
                choices.append(choice)
                
        # get pooled population estimate
        model = sm.Logit(choices, comparisons)
        result = model.fit()
        B_pooled = np.zeros(P) 
        B_pooled[EST] = result.params

        # apply move-on rule and update
        alpha = 0.05
        critical_value = norm.ppf(1 - alpha / 2)
        SE = result.bse
        sig = {EST[j] for j in np.where(np.abs(result.params/SE) > critical_value)[0]}
        S_b = {g for g, cols in enumerate(UNITS) if sig & set(cols)}
        p_flip = move_on(S_b, unit_weights(B_pooled))

        # print end of batch pooled comparisons, choices, preference:
        print("finished batch:", b)
        print("B_pooled:", B_pooled)
        print("S_b:", S_b)
        print("p_flip:", p_flip)
        print("sum(p_flip):", p_flip.sum())

In [223]:
x = np.fromiter(rand.choice(features["id"].values), dtype=int)
x

array([1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0])

In [224]:
y = rand.binomial(1, p_flip, size=P)
y

array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [225]:
x ^ y

array([0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0])

### Simulated Run:

In [136]:
B_true.shape

(17,)

In [137]:
B.shape

(150, 17)

In [151]:
run_batches(B_true, B, features, 20)

Optimization terminated successfully.
         Current function value: 0.683460
         Iterations 4
finished batch: 0
B_pooled: [ 0.         -0.12686033 -0.26845982  0.17908007  0.14250475  0.14292751
  0.05878239  0.03733646  0.34112481  0.18959726  0.02035442  0.44032253
  0.04168771  0.47819278 -0.33800219  0.14307844 -0.30691406]
S_b: {0, 6}
p_flip: [0.         0.14219585 0.11315376 0.11348945 0.04667527 0.02964646
 0.         0.15054687 0.01616212 0.34963151 0.0331015  0.37970182
 0.26838558 0.11360929 0.24370051]
sum(p_flip): 2.0000000000000004
Optimization terminated successfully.
         Current function value: 0.689246
         Iterations 4
finished batch: 1
B_pooled: [ 0.         -0.11541566 -0.2583066   0.14395203  0.02277517  0.16502682
 -0.06092714 -0.07831541  0.34526516  0.07116283  0.00297653  0.07213188
  0.23047275  0.09818458  0.02233973  0.04779034  0.04937686]
S_b: {0, 6}
p_flip: [0.         0.27022282 0.04275293 0.30978384 0.11437076 0.14701155
 0.         0.13

Optimization terminated successfully.
         Current function value: 0.692300
         Iterations 4
finished batch: 16
B_pooled: [ 0.         -0.11272583 -0.25134473  0.13631856  0.04161233  0.00226193
 -0.0060699   0.00719188  0.3328637  -0.10874971  0.00053131  0.03767428
  0.01302876  0.09071489  0.01789601  0.04944175  0.07080947]
S_b: {0, 1, 6, 7, 11, 13, 14}
p_flip: [0.         0.         0.65911961 0.03582797 0.09614429 0.11391596
 0.         0.         0.00841569 0.59674273 0.20636945 0.
 0.2834643  0.         0.        ]
sum(p_flip): 1.9999999999999998
Optimization terminated successfully.
         Current function value: 0.692374
         Iterations 4
finished batch: 17
B_pooled: [ 0.00000000e+00 -1.12506324e-01 -2.51212189e-01  1.36222811e-01
  2.94175982e-02 -3.10944138e-04  1.26417713e-03  9.09404716e-03
  3.32327593e-01 -1.08907401e-01  1.54614191e-03  3.31371477e-02
  5.99757296e-03  9.06232367e-02  1.19632345e-02  4.94658463e-02
  7.09158423e-02]
S_b: {0, 1, 6, 7, 11,